# Spatial Mapping with Synthetic Data

This notebook demonstrates NEREIDS' per-pixel isotopic composition mapping
using a **programmatically generated** 3D transmission cube — no external TIFF
files required.

We construct a 20×20 pixel "phantom" with three distinct composition zones,
simulate neutron transmission with Poisson noise, then run `spatial_map()` to
recover the isotopic density maps and compare them against the known ground truth.

| Step | NEREIDS function |
|------|------------------|
| 1. Define phantom geometry | NumPy (ground truth) |
| 2. Load nuclear data | `load_endf()` |
| 3. Generate transmission cube | `forward_model()` per unique composition |
| 4. Add Poisson noise | NumPy |
| 5. Run spatial mapping | `spatial_map()` |
| 6. Visualise composition maps | Matplotlib |
| 7. Convergence and quality | `chi_squared_map`, `converged_map` |
| 8. Quantitative validation | per-zone bias and spread |

## Prerequisites

```bash
pixi run build
```

**Previous:** [Forward Model Pipeline](03_forward_model_demo.ipynb)  
**Next:** [Enrichment Analysis](01_enrichment_analysis.ipynb)

In [ ]:
import time

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

import nereids

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

## 1. Define Phantom Geometry

Our synthetic phantom is a 20×20 pixel sample with three composition zones:

| Zone | Rows | Cols | n(U-235) [atoms/barn] | n(Pu-241) [atoms/barn] |
|------|------|------|----------------------|------------------------|
| Background | all | all | 0.0005 | 0.0002 |
| U-235-enriched band | 5–14 | 4–15 | 0.002 | 0.0002 |
| Pu-241 inclusion | 8–11 | 8–11 | 0.002 | 0.001 |

Because only three distinct compositions appear across 400 pixels, we only
need to compute three forward-model spectra rather than 400.

In [ ]:
H, W = 20, 20

# Ground-truth areal density maps (atoms/barn)
true_u235 = np.full((H, W), 0.0005)
true_pu241 = np.full((H, W), 0.0002)

# Rectangular U-235-enriched band
true_u235[5:15, 4:16] = 0.002

# Pu-241 inclusion inside the band
true_pu241[8:12, 8:12] = 0.001

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(true_u235, cmap='Blues', vmin=0, interpolation='nearest')
im1 = axes[1].imshow(true_pu241, cmap='Oranges', vmin=0, interpolation='nearest')
plt.colorbar(im0, ax=axes[0], label='n(U-235) [atoms/barn]')
plt.colorbar(im1, ax=axes[1], label='n(Pu-241) [atoms/barn]')
axes[0].set_title('Ground Truth: U-235')
axes[1].set_title('Ground Truth: Pu-241')
for ax in axes:
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

compositions = np.stack([true_u235.ravel(), true_pu241.ravel()], axis=1)
unique_comps = np.unique(compositions, axis=0)
print(f'Phantom: {H}x{W} = {H * W} pixels,  {len(unique_comps)} unique compositions')
for n_u, n_pu in unique_comps:
    print(f'  n(U-235)={n_u:.4f},  n(Pu-241)={n_pu:.4f}')

## 2. Load Nuclear Data

We use U-235 and Pu-241 — the same pair as the PLEIADES reference dataset.
Both isotopes have rich resonance spectra in the 1–50 eV range used for
VENUS measurements, enabling reliable separation by their spectral fingerprints.

In [ ]:
u235 = nereids.load_endf(92, 235)
pu241 = nereids.load_endf(94, 241)

print(u235)
print(pu241)

# 500-point grid over 1–50 eV (same as Forward Model Pipeline notebook)
energies = np.linspace(1.0, 50.0, 500)
print(f'\nEnergy grid: {len(energies)} points,  {energies[0]:.1f}–{energies[-1]:.1f} eV')

## 3. Generate Synthetic Transmission Cube

We call `forward_model()` for each of the three unique compositions and then
broadcast the results to fill the full `(n_energies, H, W)` array.

The same VENUS beamline parameters (25 m flight path, timing resolution 0.3 μs)
are used for both forward-model generation and spatial-map fitting so that
the synthetic data is self-consistent.

In [ ]:
# Identify unique (n_u235, n_pu241) pairs and their pixel assignments
compositions = np.stack([true_u235.ravel(), true_pu241.ravel()], axis=1)
unique_comps, pixel_indices = np.unique(compositions, axis=0, return_inverse=True)

print(f'Computing {len(unique_comps)} unique spectra for {H * W} pixels ...')

# VENUS beamline parameters (shared with spatial_map below)
FLIGHT_PATH_M = 25.0
DELTA_T_US = 0.3
DELTA_L_M = 0.01
TEMP_K = 293.6

unique_spectra = {}
for i, (n_u, n_pu) in enumerate(unique_comps):
    t = nereids.forward_model(
        energies,
        [(u235, float(n_u)), (pu241, float(n_pu))],
        temperature_k=TEMP_K,
        flight_path_m=FLIGHT_PATH_M,
        delta_t_us=DELTA_T_US,
        delta_l_m=DELTA_L_M,
    )
    unique_spectra[i] = t
    print(f'  Composition {i}: n(U-235)={n_u:.4f}, n(Pu-241)={n_pu:.4f}')

# Assemble (n_energies, H, W) cube by broadcasting pixel-indexed spectra
# np.stack([1D_arrays...], axis=1) → (n_energies, n_pixels), then reshape
transmission_true = np.stack(
    [unique_spectra[idx] for idx in pixel_indices], axis=1
).reshape(len(energies), H, W)

print(f'\nTransmission cube shape: {transmission_true.shape}')
print(f'  min T = {transmission_true.min():.4f},  max T = {transmission_true.max():.4f}')

## 4. Add Poisson Noise

Real VENUS measurements have Poisson counting statistics:
$$N(E) \sim \mathrm{Poisson}\bigl(I_0 \cdot T(E)\bigr)$$

where $I_0$ is the open-beam intensity per energy bin. We use $I_0 = 300$,
a typical low-count condition at VENUS. The corresponding Poisson uncertainty
on the measured transmission is $\sigma_T = \sqrt{N} / I_0$.

In [ ]:
I0 = 300  # open-beam counts per energy bin
rng = np.random.default_rng(42)

# Draw Poisson counts; clip to >=1 to avoid zero-uncertainty bins
counts = rng.poisson(I0 * transmission_true)
counts = np.maximum(counts, 1)

transmission_noisy = counts / I0
uncertainty = np.sqrt(counts) / I0  # Poisson: sigma_T = sqrt(N) / I0

print(f'Mean transmission:         {transmission_noisy.mean():.4f}')
print(f'Mean relative uncertainty: {(uncertainty / transmission_noisy).mean() * 100:.1f}%')

# Show one energy slice at E ~ 8.5 eV to verify phantom geometry survives noise
e_idx = 150
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
data_rows = [
    (transmission_true[e_idx],  'True T',            'viridis', 0.0, 1.0),
    (transmission_noisy[e_idx], f'Noisy T  (I0={I0})', 'viridis', 0.0, 1.0),
    (uncertainty[e_idx],        'Uncertainty sigma',  'Oranges', 0.0, None),
]
for ax, (data, title, cmap, vmin, vmax) in zip(axes, data_rows):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{title}  (E={energies[e_idx]:.1f} eV)')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

## 5. Run Spatial Mapping

`spatial_map()` fits every pixel independently in parallel (Rayon) using
Levenberg-Marquardt optimization. The fitting model applies the same Doppler
and instrument-resolution broadening as the forward model above.

The returned `SpatialResult` contains density maps, uncertainty maps,
convergence flags, and reduced chi-squared values for all pixels.

In [ ]:
t0 = time.perf_counter()

result = nereids.spatial_map(
    transmission_noisy,
    uncertainty,
    energies,
    [u235, pu241],
    temperature_k=TEMP_K,
    initial_densities=[0.001, 0.0005],
    flight_path_m=FLIGHT_PATH_M,
    delta_t_us=DELTA_T_US,
    delta_l_m=DELTA_L_M,
    max_iter=150,
)

elapsed = time.perf_counter() - t0

print(f'Spatial mapping: {elapsed:.2f} s  ({H * W} pixels, {len(energies)} energies each)')
print(f'Converged: {result.n_converged}/{result.n_total}  ({result.n_converged / result.n_total * 100:.1f}%)')
print(f'Isotope names: {result.isotope_names}')
print()

# Convert list of 2D arrays to a single 3D array for easier indexing
density_maps = np.array(result.density_maps)      # (n_isotopes, H, W)
unc_maps = np.array(result.uncertainty_maps)       # (n_isotopes, H, W)
chi2 = np.array(result.chi_squared_map)            # (H, W)
conv = np.array(result.converged_map)              # (H, W) bool

print(f'density_maps shape:     {density_maps.shape}')
print(f'uncertainty_maps shape: {unc_maps.shape}')
print(f'chi_squared_map shape:  {chi2.shape}')
print(f'converged_map shape:    {conv.shape}')

## 6. Visualise Composition Maps

For each isotope we show three panels side-by-side:
- **Ground truth** — the known input density
- **Fitted** — what `spatial_map()` recovered
- **Bias** (fitted − true) — systematic error; red = over-estimate, blue = under-estimate

In [ ]:
names = result.isotope_names
truths = [true_u235, true_pu241]
cmaps = ['Blues', 'Oranges']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for row_i, (name, truth, fitted, cmap) in enumerate(zip(names, truths, density_maps, cmaps)):
    vmax = truth.max() * 1.15
    bias = fitted - truth
    vlim = np.abs(bias).max()

    im0 = axes[row_i, 0].imshow(truth,  cmap=cmap,    vmin=0,     vmax=vmax, interpolation='nearest')
    im1 = axes[row_i, 1].imshow(fitted, cmap=cmap,    vmin=0,     vmax=vmax, interpolation='nearest')
    im2 = axes[row_i, 2].imshow(bias,   cmap='RdBu_r', vmin=-vlim, vmax=vlim, interpolation='nearest')

    plt.colorbar(im0, ax=axes[row_i, 0], label='[atoms/barn]')
    plt.colorbar(im1, ax=axes[row_i, 1], label='[atoms/barn]')
    plt.colorbar(im2, ax=axes[row_i, 2], label='bias [atoms/barn]')

    axes[row_i, 0].set_title(f'{name}  —  Ground Truth')
    axes[row_i, 1].set_title(f'{name}  —  Fitted')
    axes[row_i, 2].set_title(f'{name}  —  Bias (fitted - true)')

    for ax in axes[row_i]:
        ax.set_xlabel('Column')
        ax.set_ylabel('Row')

plt.tight_layout()
plt.show()

## 7. Convergence and Quality Metrics

- **Reduced χ²** — values near 1 indicate a statistically sound fit.
  Values > 5 suggest either poor convergence or model mismatch.
- **Convergence map** — LM is declared converged when the parameter
  update norm falls below the tolerance; non-converged pixels (red)
  should be investigated or re-fitted with relaxed settings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(chi2, cmap='RdYlGn_r', vmin=0, vmax=5, interpolation='nearest')
plt.colorbar(im0, ax=axes[0], label='Reduced chi^2')
axes[0].set_title('Reduced chi^2 Map')
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row')

cmap_conv = mcolors.ListedColormap(['tomato', 'limegreen'])
im1 = axes[1].imshow(
    conv.astype(int), cmap=cmap_conv, vmin=0, vmax=1, interpolation='nearest'
)
plt.colorbar(im1, ax=axes[1], label='Converged (1 = yes)')
axes[1].set_title('Convergence Map')
axes[1].set_xlabel('Column')
axes[1].set_ylabel('Row')

plt.tight_layout()
plt.show()

print(f'Reduced chi^2  —  mean: {chi2.mean():.3f},  median: {np.median(chi2):.3f},  max: {chi2.max():.3f}')
print(f'Converged: {conv.sum()} / {conv.size} pixels  ({conv.mean() * 100:.1f}%)')

## 8. Quantitative Validation

For each composition zone we compute the mean and standard deviation of the
fitted densities and compare against ground truth.
The **bias** is `(mean_fit − true) / true × 100 %` — a measure of systematic error.
The **spread** reflects Poisson noise propagation through the fit.

In [ ]:
zones = [
    ('Background',  (true_u235 == 0.0005) & (true_pu241 == 0.0002), 0.0005, 0.0002),
    ('U-235 band',  (true_u235 == 0.002)  & (true_pu241 == 0.0002), 0.002,  0.0002),
    ('Pu-241 core', (true_u235 == 0.002)  & (true_pu241 == 0.001),  0.002,  0.001),
]

header = f'{"Zone":<16} {"Isotope":<10} {"True":>10} {"Mean fit":>10} {"Std":>10} {"Bias":>8}'
print(header)
print('-' * len(header))

for zone_name, mask, true_u, true_pu in zones:
    for iso_idx, (iso_name, true_n) in enumerate([('U-235', true_u), ('Pu-241', true_pu)]):
        vals = density_maps[iso_idx][mask]
        mean_fit = vals.mean()
        std_fit = vals.std()
        bias_pct = (mean_fit - true_n) / true_n * 100
        print(
            f'{zone_name:<16} {iso_name:<10} {true_n:>10.4f}'
            f' {mean_fit:>10.6f} {std_fit:>10.6f} {bias_pct:>7.1f}%'
        )
    print()

## Summary

This notebook demonstrated end-to-end per-pixel isotopic mapping:

1. **Phantom** — 20×20 pixel synthetic object with three U-235/Pu-241 composition zones
2. **Efficient cube generation** — only three unique `forward_model()` calls needed for 400 pixels
3. **Poisson noise** — simulates VENUS low-count conditions (I₀ = 300)
4. **`spatial_map()`** — parallel per-pixel LM fitting recovers composition maps in seconds
5. **Validation** — biases are typically < 5% even at low count rates

### `SpatialResult` at a Glance

| Attribute | Shape | Description |
|-----------|-------|-------------|
| `density_maps` | list of `(H, W)` | one 2D array per isotope |
| `uncertainty_maps` | list of `(H, W)` | Cramér–Rao lower bounds |
| `chi_squared_map` | `(H, W)` | reduced χ² per pixel |
| `converged_map` | `(H, W)` bool | LM convergence flag |
| `isotope_names` | `[str, …]` | auto-labelled from ENDF data |

### Quick-start pattern

```python
import nereids, numpy as np

u235  = nereids.load_endf(92, 235)
pu241 = nereids.load_endf(94, 241)

result = nereids.spatial_map(
    transmission,           # (n_e, H, W) float64
    uncertainty,            # (n_e, H, W) float64
    energies,               # (n_e,) eV
    [u235, pu241],
    temperature_k=293.6,
    flight_path_m=25.0,
    delta_t_us=0.3,
    delta_l_m=0.01,
)

density_maps = np.array(result.density_maps)   # (2, H, W)
```

**Previous:** [Forward Model Pipeline](03_forward_model_demo.ipynb)  
**Next:** [Enrichment Analysis](01_enrichment_analysis.ipynb)